# 🏎️ RAG Testing — F1 Race Strategist AI

This notebook validates the retrieval pipeline against **5 historical races**
with known outcomes (the "ground truth"). For each race we verify:

1. **Winner** — correct driver abbreviation
2. **Strategy type** — 1-stop, 2-stop, etc.
3. **Compound sequence** — exact tire order used by the winner
4. **Context completeness** — all required sections are present
5. **Key facts** — weather, safety cars, notable events

### Races selected
| # | Race | Year | Winner | Strategy | Why chosen |
|---|------|------|--------|----------|------------|
| 1 | Silverstone | 2022 | SAI | Multi-stop (red flag + SC) | Chaotic race, Sainz 1st win |
| 2 | Monaco | 2023 | VER | 1-stop (M→I) | Rain crossover, long stint |
| 3 | Bahrain | 2023 | VER | 2-stop (S→S→H) | Season opener, high degradation |
| 4 | Hungary | 2023 | VER | 1-stop (M→H) | Classic 1-stop, hot conditions |
| 5 | São Paulo | 2022 | RUS | 2-stop (S→M→S) | Russell 1st win, safety car |


## 0. Setup

In [ ]:
import sys, os, logging, json
from dataclasses import dataclass, field, asdict
from typing import Optional

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath('..'))
logging.basicConfig(level=logging.WARNING)


## 1. Ground-Truth Definitions

Each `GroundTruth` encodes the **known facts** we expect the RAG pipeline to return.

In [ ]:
@dataclass
class GroundTruth:
    """Known race outcome for validation."""
    name: str
    circuit: str
    year: int
    winner: str                    # 3-letter abbreviation
    winner_team: str
    podium: list[str]              # top-3 abbreviations
    strategy_type: str             # '1-stop', '2-stop', etc.
    winner_compounds: list[str]    # ordered compound list
    total_laps: int
    rainfall: bool
    safety_car: bool
    red_flag: bool
    key_facts: list[str] = field(default_factory=list)
    notes: str = ''


In [ ]:
GROUND_TRUTH: list[GroundTruth] = [
    # ── 1. Silverstone 2022 ─────────────────────────────────────
    GroundTruth(
        name='British GP 2022',
        circuit='Silverstone',
        year=2022,
        winner='SAI',
        winner_team='Ferrari',
        podium=['SAI', 'PER', 'HAM'],
        strategy_type='multi-stop',
        winner_compounds=['SOFT', 'HARD', 'MEDIUM', 'SOFT'],
        total_laps=52,
        rainfall=False,
        safety_car=True,
        red_flag=True,
        key_facts=[
            'Lap-1 red flag after Zhou Guanyu crash',
            'Late safety car — Sainz pitted for fresh softs, Leclerc stayed out',
            'Carlos Sainz first career victory',
        ],
        notes='Red flag restart shuffled strategies; late SC was decisive.',
    ),

    # ── 2. Monaco 2023 ──────────────────────────────────────────
    GroundTruth(
        name='Monaco GP 2023',
        circuit='Monaco',
        year=2023,
        winner='VER',
        winner_team='Red Bull Racing',
        podium=['VER', 'ALO', 'OCO'],
        strategy_type='1-stop',
        winner_compounds=['MEDIUM', 'INTERMEDIATE'],
        total_laps=78,
        rainfall=True,
        safety_car=False,
        red_flag=False,
        key_facts=[
            'Verstappen did 55 laps on mediums before rain arrived',
            'Late rain forced switch to intermediates',
            'Graining management was key in first stint',
        ],
        notes='Extreme tyre management race; weather crossover.',
    ),

    # ── 3. Bahrain 2023 ─────────────────────────────────────────
    GroundTruth(
        name='Bahrain GP 2023',
        circuit='Bahrain',
        year=2023,
        winner='VER',
        winner_team='Red Bull Racing',
        podium=['VER', 'PER', 'ALO'],
        strategy_type='2-stop',
        winner_compounds=['SOFT', 'SOFT', 'HARD'],
        total_laps=57,
        rainfall=False,
        safety_car=False,
        red_flag=False,
        key_facts=[
            'Season opener 2023',
            'Red Bull used double-soft opening stints',
            'Verstappen won by ~12 seconds',
            'High rear tyre degradation on Sakhir circuit',
        ],
        notes='Desert night race; high degradation favoured 2-stop.',
    ),

    # ── 4. Hungary 2023 ─────────────────────────────────────────
    GroundTruth(
        name='Hungarian GP 2023',
        circuit='Budapest',
        year=2023,
        winner='VER',
        winner_team='Red Bull Racing',
        podium=['VER', 'NOR', 'PER'],
        strategy_type='1-stop',
        winner_compounds=['MEDIUM', 'HARD'],
        total_laps=70,
        rainfall=False,
        safety_car=False,
        red_flag=False,
        key_facts=[
            'Verstappen pitted on lap 23 for hards',
            'Dominant drive, led from start',
            'Classic 1-stop at a low-degradation twisty circuit',
        ],
        notes='Clean dry race; textbook 1-stop strategy.',
    ),

    # ── 5. São Paulo 2022 ───────────────────────────────────────
    GroundTruth(
        name='São Paulo GP 2022',
        circuit='São Paulo',
        year=2022,
        winner='RUS',
        winner_team='Mercedes',
        podium=['RUS', 'HAM', 'SAI'],
        strategy_type='2-stop',
        winner_compounds=['SOFT', 'MEDIUM', 'SOFT'],
        total_laps=71,
        rainfall=False,
        safety_car=True,
        red_flag=False,
        key_facts=[
            'George Russell first career victory',
            'Mercedes 1-2 finish (Russell–Hamilton)',
            'Safety car played into Mercedes strategy',
        ],
        notes='Russell maiden win; 2-stop soft-medium-soft was standard.',
    ),
]

print(f'✅ {len(GROUND_TRUTH)} ground-truth scenarios loaded:')
for gt in GROUND_TRUTH:
    print(f'   🏁 {gt.name}: {gt.winner} ({gt.strategy_type})')


## 2. Embedder — Collection Health Check

In [ ]:
from rag.embedder import get_collection, list_collections

collections = list_collections()
print("📦 Collections in ChromaDB:")
for c in collections:
    print(f"  • {c['name']}: {c['count']} documents")

col = get_collection("f1_circuits")
assert col.count() > 0, "f1_circuits is empty — run `python -m rag.ingestor` first!"
print(f"\n✅ f1_circuits: {col.count()} documents")


## 3. Circuit Vector Search

Verify that the vector search returns the correct circuit for each ground-truth race.

In [ ]:
from rag.retriever import retrieve_circuits

print('🔍 Circuit retrieval accuracy test\n')
circuit_hits = 0

for gt in GROUND_TRUTH:
    query = f'race circuit {gt.circuit}'
    results = retrieve_circuits(query, n_results=3)
    top_name = results[0]['metadata']['name'] if results else 'N/A'
    match = gt.circuit.lower() in top_name.lower()
    status = '✅' if match else '❌'
    if match:
        circuit_hits += 1
    print(f'  {status} {gt.name}: query="{query}" → top result="{top_name}"')

accuracy = circuit_hits / len(GROUND_TRUTH) * 100
print(f'\n📊 Circuit retrieval accuracy: {circuit_hits}/{len(GROUND_TRUTH)} ({accuracy:.0f}%)')
assert circuit_hits >= 3, f'Circuit accuracy too low: {circuit_hits}/{len(GROUND_TRUTH)}'


## 4. Full Race Context Retrieval

The core test: call `retrieve_race_context` for each ground-truth race
and validate winner, strategy, and context completeness.

In [ ]:
from rag.retriever import retrieve_race_context
import pandas as pd

race_results: list[dict] = []

for gt in GROUND_TRUTH:
    print('=' * 65)
    print(f'🏁 {gt.name} — Expected winner: {gt.winner}')
    print('=' * 65)

    result = {
        'race': gt.name,
        'winner_ok': False,
        'strategy_ok': False,
        'context_ok': False,
        'error': None,
    }

    try:
        ctx = retrieve_race_context(
            query=f'race strategy {gt.circuit} {gt.year}',
            year=gt.year,
            circuit=gt.circuit,
        )

        if ctx['error']:
            print(f'  ⚠️ Error: {ctx["error"]}')
            result['error'] = ctx['error']
            race_results.append(result)
            print()
            continue

        # ── Winner validation ───────────────────────────────────
        results_df = ctx['race_results']
        actual_winner = results_df.iloc[0]['Abbreviation']
        result['winner_ok'] = actual_winner == gt.winner
        w_status = '✅' if result['winner_ok'] else '❌'
        print(f'  {w_status} Winner: {actual_winner} (expected {gt.winner})')

        # ── Strategy validation ─────────────────────────────────
        stints = ctx['winner_stints']
        if stints is not None and not stints.empty:
            actual_compounds = list(stints['Compound'])
            num_stops = len(actual_compounds) - 1
            print(f'  📋 Stints: {" → ".join(actual_compounds)} ({num_stops}-stop)')
            print(f'  📋 Expected: {" → ".join(gt.winner_compounds)} ({gt.strategy_type})')
            # Check compound match (flexible — order matters)
            result['strategy_ok'] = (
                [c.upper() for c in actual_compounds] ==
                [c.upper() for c in gt.winner_compounds]
            )
            s_status = '✅' if result['strategy_ok'] else '⚠️'
            print(f'  {s_status} Compound match: {result["strategy_ok"]}')
        else:
            print('  ❌ No stint data returned')

        # ── Context completeness ────────────────────────────────
        text = ctx['context_text']
        required = ['Race Context', 'Race Results', 'Winning Strategy',
                     'Top-5 Strategies', 'Weather']
        missing = [s for s in required if s not in text]
        result['context_ok'] = len(missing) == 0
        c_status = '✅' if result['context_ok'] else '❌'
        print(f'  {c_status} Context sections: {len(required)-len(missing)}/{len(required)}')
        if missing:
            print(f'      Missing: {missing}')
        print(f'  📏 Context length: {len(text)} chars')

    except Exception as exc:
        print(f'  💥 Exception: {exc}')
        result['error'] = str(exc)

    race_results.append(result)
    print()


## 5. Results Summary

Aggregate pass/fail across all ground-truth scenarios.

In [ ]:
print('=' * 65)
print('📊 GROUND-TRUTH VALIDATION SUMMARY')
print('=' * 65)

total = len(race_results)
winners_ok = sum(1 for r in race_results if r['winner_ok'])
strategy_ok = sum(1 for r in race_results if r['strategy_ok'])
context_ok = sum(1 for r in race_results if r['context_ok'])
errors = sum(1 for r in race_results if r['error'])

print(f'\n  Races tested:       {total}')
print(f'  Winner correct:     {winners_ok}/{total} ({winners_ok/total*100:.0f}%)')
print(f'  Strategy match:     {strategy_ok}/{total} ({strategy_ok/total*100:.0f}%)')
print(f'  Context complete:   {context_ok}/{total} ({context_ok/total*100:.0f}%)')
print(f'  Errors:             {errors}/{total}')

print(f'\n  Per-race breakdown:')
for r in race_results:
    w = '✅' if r['winner_ok'] else '❌'
    s = '✅' if r['strategy_ok'] else '❌'
    c = '✅' if r['context_ok'] else '❌'
    err = f' ⚠️ {r["error"][:40]}...' if r['error'] else ''
    print(f'    {r["race"]:.<30} W:{w} S:{s} C:{c}{err}')

# Overall pass/fail
all_pass = winners_ok == total and context_ok == total
print(f'\n{"✅ ALL CRITICAL CHECKS PASSED" if all_pass else "⚠️ SOME CHECKS FAILED — review above"}')


## 6. Deep Dive — Silverstone 2022

Detailed validation of the chaotic British GP:
Sainz's first win, red flag on lap 1, late safety car.

In [ ]:
gt = GROUND_TRUTH[0]  # Silverstone 2022

ctx = retrieve_race_context(
    query='strategy at Silverstone 2022',
    year=2022,
    circuit='Silverstone',
)

assert ctx['error'] is None, f'Retrieval failed: {ctx["error"]}'

results_df = ctx['race_results']
winner = results_df.iloc[0]['Abbreviation']
print(f'🏆 Winner: {winner}')
assert winner == 'SAI', f'Expected SAI, got {winner}'

stints = ctx['winner_stints']
compounds = list(stints['Compound'])
print(f'🔧 Strategy: {" → ".join(compounds)}')
assert len(compounds) >= 3, f'Expected 3+ stints, got {len(compounds)}'

weather = ctx['weather']
assert not weather.empty, 'Weather data should not be empty'
print(f'🌡️ Weather points: {len(weather)}')

# Podium check
top3 = list(results_df.head(3)['Abbreviation'])
print(f'🥇🥈🥉 Podium: {top3}')
assert top3 == gt.podium, f'Expected {gt.podium}, got {top3}'

print('\n✅ Silverstone 2022 deep-dive passed!')


## 7. Deep Dive — Monaco 2023

Rain crossover race: Verstappen did 55 laps on mediums before
switching to intermediates when rain arrived.

In [ ]:
gt = GROUND_TRUTH[1]  # Monaco 2023

ctx = retrieve_race_context(
    query='Monaco GP strategy 2023',
    year=2023,
    circuit='Monaco',
)

assert ctx['error'] is None, f'Retrieval failed: {ctx["error"]}'

results_df = ctx['race_results']
winner = results_df.iloc[0]['Abbreviation']
print(f'🏆 Winner: {winner}')
assert winner == 'VER', f'Expected VER, got {winner}'

stints = ctx['winner_stints']
compounds = list(stints['Compound'])
print(f'🔧 Strategy: {" → ".join(compounds)}')

# Check weather data shows rainfall
weather = ctx['weather']
if not weather.empty and 'Rainfall' in weather.columns:
    had_rain = weather['Rainfall'].any()
    rain_status = '✅' if had_rain == gt.rainfall else '⚠️'
    print(f'{rain_status} Rainfall detected: {had_rain} (expected {gt.rainfall})')

# The first stint should be very long (50+ laps on MEDIUM)
if not stints.empty:
    first_stint_laps = int(stints.iloc[0]['Laps'])
    long_stint = first_stint_laps >= 45
    ls_status = '✅' if long_stint else '⚠️'
    print(f'{ls_status} First stint length: {first_stint_laps} laps (expected ≥45)')

print('\n✅ Monaco 2023 deep-dive passed!')


## 8. Context Text Quality

Verify that every assembled context string contains all required sections.

In [ ]:
print('📋 Context structure validation across all races\n')
required_sections = ['Race Context', 'Race Results', 'Winning Strategy',
                     'Top-5 Strategies', 'Weather']

all_ok = True
for gt in GROUND_TRUTH:
    try:
        ctx = retrieve_race_context(
            query=f'strategy {gt.circuit}',
            year=gt.year,
            circuit=gt.circuit,
        )
        if ctx['error']:
            print(f'  ⚠️ {gt.name}: {ctx["error"]}')
            all_ok = False
            continue

        text = ctx['context_text']
        missing = [s for s in required_sections if s not in text]
        status = '✅' if not missing else '❌'
        if missing:
            all_ok = False
        print(f'  {status} {gt.name}: {len(required_sections)-len(missing)}/{len(required_sections)} sections, {len(text)} chars')
        if missing:
            print(f'      Missing: {missing}')
    except Exception as exc:
        print(f'  💥 {gt.name}: {exc}')
        all_ok = False

print(f'\n{"✅ All contexts well-formed!" if all_ok else "⚠️ Some contexts have issues"}')


## 9. Export Ground Truth (JSON)

Save ground-truth data as JSON for downstream pipeline tests.

In [ ]:
import json
from dataclasses import asdict

gt_export = [asdict(gt) for gt in GROUND_TRUTH]

output_path = os.path.join('data', 'ground_truth_races.json')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(gt_export, f, indent=2, ensure_ascii=False)

print(f'💾 Ground truth exported to {output_path}')
print(f'   {len(gt_export)} races, sample keys: {list(gt_export[0].keys())}')
